# Prophet + Quantum ML + QAOA on stock (ETF) data

**IQ Hackathon Fall 2026 (Beginners) — project notebook.** Combines the three Training
Session II notebooks on one real problem:

| notebook | used here for |
|---|---|
| `QML_1_Lecture.ipynb` (quantum kernels, ZZ feature map, QSVM, VQC) | ranking 18 ETFs for next month from Prophet + momentum features |
| `QAOA_ibm_notebook.ipynb` (max-cut -> Ising -> QAOA) | picking a 4-ETF portfolio from an 18-qubit QUBO |
| `vqe_lecture.ipynb` (variational loop) | the same classical-optimiser-in-the-loop idea drives the VQC and QAOA |

Plus **Prophet** (Meta's time-series forecaster) as the feature generator, and a
**recursive self-improvement** loop that tries to make Prophet itself better.

Everything was **pre-registered** (`PREREG.md`, sha256 in `PREREG.sha256`) before any
result existed, and evaluated walk-forward on 2014-01..2026-08 with no look-ahead.
Heavy runs live in the `run_*.py` scripts; this notebook loads their results and
re-runs small pieces live. *Paper backtest only — not investment advice.*

In [ ]:
# Install every package this notebook needs (fresh environment or Google Colab).
# In Colab, first get the code and data:
#   !git clone https://github.com/Futuremine97/IQ_Hackathon_Fall_2026.git
#   %cd IQ_Hackathon_Fall_2026
%pip install -q -r requirements.txt

In [1]:
import json, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import logging; [logging.getLogger(n).setLevel(logging.ERROR) for n in ("cmdstanpy", "prophet")]
from qps.data import load_prices, UNIVERSE
from qps.panel import build_panel
from qps.quantum import zz_states, gram, bit_matrix
R = lambda f: json.load(open(os.path.join("results", f)))
px = load_prices()          # frozen file, sha256-checked
print(px.shape, px.index[0].date(), "->", px.index[-1].date())
(px / px.iloc[0]).plot(figsize=(10, 4), legend=False, logy=True, title="18 ETFs, growth of $1 (log scale)"); plt.show()

(5482, 18) 2005-01-03 -> 2026-10-16


![output](figures/notebook/cell01_1.png)

## 1. Prophet forecast (same configuration as our earlier forecaster)
`Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True,
changepoint_prior_scale=0.05, interval_width=0.8)` on the last 730 days of closes,
30-day horizon. The signal rule is BULLISH if the forecast is >= +3 %, BEARISH if <= -3 %.

In [2]:
from prophet import Prophet
t = pd.Timestamp("2026-09-30"); s = px["SPY"].loc[t - pd.Timedelta(days=730):t]
m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True,
            changepoint_prior_scale=0.05, interval_width=0.8).fit(pd.DataFrame({"ds": s.index, "y": s.values}))
fc = m.predict(m.make_future_dataframe(periods=30))
ax = plt.figure(figsize=(10, 4)).gca()
ax.plot(s.index, s.values, color="k", lw=1, label="SPY close")
ax.plot(fc.ds, fc.yhat, label="Prophet yhat"); ax.fill_between(fc.ds, fc.yhat_lower, fc.yhat_upper, alpha=.2, label="80 % interval")
real = px["SPY"].loc[t:t + pd.Timedelta(days=30)]; ax.plot(real.index, real.values, color="crimson", lw=2, label="what happened")
ax.axvline(t, ls="--", c="gray"); ax.legend(); ax.set_title("SPY: Prophet 30-day forecast from 2026-09-30"); plt.show()

![output](figures/notebook/cell03_3.png)

## 2. The prediction task
Every month-end *t* and every ETF we compute 4 features, all from data up to *t*:
Prophet's 30-day forecast return `p_ret`, Prophet's in-sample residual `p_resid`,
12-1 month momentum and 21-day volatility. Target: will this ETF finish **above the
median of the 18 next month** (exactly 9 winners per month, so a coin scores 50 %).
Features are converted to cross-sectional ranks in (0,1) -> 4 numbers -> **4 qubits**.

In [3]:
_, df, U = build_panel()
df[["date", "asset", "p_ret", "p_resid", "mom_12_1", "vol_21", "fwd_ret", "y"]].tail(18).round(4)

,date,asset,p_ret,p_resid,mom_12_1,vol_21,fwd_ret,y
4248,2026-09-30,XLK,0.0074,0.0085,0.3358,0.0116,NaN,NaN
4249,2026-09-30,XLF,0.0387,-0.0401,0.0791,0.0086,NaN,NaN
4250,2026-09-30,XLE,0.0756,-0.0407,0.4484,0.0120,NaN,NaN
4251,2026-09-30,XLV,0.0535,-0.0217,0.2710,0.0088,NaN,NaN
4252,2026-09-30,XLI,-0.0095,-0.0192,0.1542,0.0082,NaN,NaN
4253,2026-09-30,XLP,0.0176,-0.0188,0.1105,0.0071,NaN,NaN
4254,2026-09-30,XLY,-0.0294,-0.0251,-0.0257,0.0098,NaN,NaN
4255,2026-09-30,XLU,0.0204,-0.0404,-0.0095,0.0090,NaN,NaN
4256,2026-09-30,XLB,-0.0030,-0.0167,0.1975,0.0090,NaN,NaN
4257,2026-09-30,SPY,0.0076,-0.0078,0.1653,0.0068,NaN,NaN


## 3. Quantum kernel (QML notebook, Parts 2-5)
ZZ feature map with the notebook's product data map, scaled by a bandwidth lambda
(Part 5: large lambda -> exponential kernel concentration). Our batched statevector
kernel is checked against `FidelityStatevectorKernel` to 1e-15 (`results/CONTROLS_SIM.json`).

In [4]:
from qiskit.circuit.library import zz_feature_map
product_map = lambda x: x[0] if len(x) == 1 else x[0] * x[1]
display(zz_feature_map(4, reps=2, data_map_func=product_map).decompose().draw("mpl", fold=-1))
sel = R("DEV_SELECTION.json")["selected"]; lam = sel["qsvm"][0]
idx = np.where((df.date >= "2013-01-01") & (df.date <= "2013-12-31"))[0]
order = idx[np.argsort(df.y.values[idx], kind="stable")]
Kq = gram(zz_states(lam*np.pi*U[order]), zz_states(lam*np.pi*U[order]))
Kr = np.exp(-sel["rbf"][0] * ((U[order][:, None] - U[order][None]) ** 2).sum(-1))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, K, tl in [(ax[0], Kq, f"quantum ZZ kernel (lambda={lam})"), (ax[1], Kr, f"RBF kernel (gamma={sel['rbf'][0]})")]:
    a.imshow(K, cmap="viridis"); a.set_title(tl + "\nsorted: losers | winners")
for l in [0.1, 0.3, 0.5, 1.0]:
    K = gram(zz_states(l*np.pi*U[idx]), zz_states(l*np.pi*U[idx])); ax[2].hist(K[np.triu_indices_from(K, 1)], bins=40, alpha=.5, label=f"lambda={l}")
ax[2].set_title("off-diagonal kernel values"); ax[2].legend(); plt.show()

![output](figures/notebook/cell07_0.png)

![output](figures/notebook/cell07_1.png)

## 4. Walk-forward results (152 test months x 18 ETFs)
Hyper-parameters were chosen on 2009-2013 only. Arms: Prophet rule, momentum rule,
QSVM, VQC, RBF-SVM, logistic regression.

In [5]:
res = R("RESULT.json"); arms = pd.DataFrame(res["arms"]).T
display(arms.round(3))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(arms.index, arms.acc_pct, yerr=1.96 * arms.acc_se_pct, color=["#888", "#888", "#7b3fe4", "#3f8fe4", "#3f8fe4", "#7b3fe4"])
ax[0].axhline(50, c="k", ls="--"); ax[0].set_ylim(44, 58); ax[0].set_ylabel("top-half accuracy %"); ax[0].tick_params(axis="x", rotation=30)
ts = pd.read_csv("results/test_scores.csv", parse_dates=["date"])
from qps.panel import top9_accuracy
for col, c in [("qsvm", "#7b3fe4"), ("rbf", "#3f8fe4"), ("p_ret", "#888")]:
    a = ts.groupby("date").apply(lambda d: top9_accuracy(d[col].values, d.y.values))
    ax[1].plot(a.index, a.expanding().mean() * 100, label=col, color=c)
ax[1].axhline(50, c="k", ls="--"); ax[1].legend(); ax[1].set_title("running accuracy"); plt.show()
print("H1 Prophet channel:", res["H1"]); print("H2 QSVM vs RBF:", res["H2"])
print("controls VOID:", res["controls_void"], "| planted-signal check:", R("CONTROLS.json")["PC3_planted_acc"])

,acc_pct,acc_se_pct,mean_ic,ic_t,ic_p
M0_prophet,50.073,1.210,0.007,0.245,0.807
M5_momentum,51.681,1.361,0.030,1.024,0.307
M1_qsvm,49.708,0.861,0.009,0.446,0.657
M3_rbf,52.266,1.052,0.021,0.988,0.324
M4_logit,51.389,1.328,0.022,0.763,0.447
M2_vqc,49.561,0.967,-0.015,-0.796,0.427


![output](figures/notebook/cell09_1.png)

H1 Prophet channel: {'mean_ic': 0.006925207756232685, 'ic_p': 0.8065144480288637, 'acc_pct': 50.07309941520468, 'shuffle_band_95_pct': [47.514619883040936, 51.97551169590643], 'verdict': 'CLOSED-NEG'}
H2 QSVM vs RBF: {'gain_pt': -2.558479532163743, 'p_signflip': 0.050794920507949204, 'verdict': 'CLOSED-NEG'}
controls VOID: False | planted-signal check: {'qsvm': 0.6016081871345029, 'rbf': 0.7865497076023391}


## 5. QAOA portfolio (QAOA notebook, max-cut -> portfolio QUBO)
Each month: choose exactly K=4 of 18 ETFs to minimise
`-mu.x + (1/8) x^T S x + A (sum x - 4)^2` (mu = QSVM scores, S = 1-year covariance).
First a small **8-ETF instance with Qiskit's own `QAOAAnsatz` + `StatevectorEstimator`**,
exactly as in the training notebook; then the full 18-qubit backtest.

In [6]:
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from scipy.optimize import minimize
from run_qaoa_portfolio import build_cost
sub = UNIVERSE[:8]; t = ts.date.max(); d = ts[ts.date == t].set_index("asset").loc[sub]
mu = ((d.qsvm - d.qsvm.mean()) / d.qsvm.std()).values
S = px[sub].pct_change().loc[:t].iloc[-252:].cov().to_numpy(copy=True); S = S / np.mean(np.diag(S))
B = bit_matrix(8).astype(float); E, _ = build_cost(mu, S, B); E = (E - E.mean()) / E.std()
# exact Pauli-Z expansion of the diagonal cost (Walsh transform), as SparsePauliOp
terms = []
for Sm in range(256):
    sel_ = np.array([(Sm >> i) & 1 for i in range(8)]); c = float(E @ ((-1.0) ** (B @ sel_))) / 256
    if abs(c) > 1e-12: terms.append(("".join("Z" if sel_[7 - q] else "I" for q in range(8)), c))
H = SparsePauliOp.from_list(terms); ans = QAOAAnsatz(H, reps=2)
est = StatevectorEstimator()
f = lambda x: float(est.run([(ans, H, x)]).result()[0].data.evs)
opt = minimize(f, [0.4, 0.2, 0.2, 0.4], method="COBYLA", options={"maxiter": 200})
circ = ans.assign_parameters(opt.x); circ.measure_all()
counts = StatevectorSampler(seed=1).run([circ], shots=2000).result()[0].data.meas.get_counts()
best = int(np.argmin(E)); top = sorted(counts.items(), key=lambda kv: -kv[1])[:8]
print("brute-force optimum:", [sub[i] for i in range(8) if (best >> i) & 1])
for bits, n in top:
    b = int(bits, 2); print(f"{bits}  {n:4d} shots  {'<- optimum' if b == best else ''}  {[sub[i] for i in range(8) if (b >> i) & 1]}")

brute-force optimum: ['XLF', 'XLE', 'XLV', 'XLY']
00111100    30 shots    ['XLE', 'XLV', 'XLI', 'XLP']
10110100    26 shots    ['XLE', 'XLI', 'XLP', 'XLU']
10010110    26 shots    ['XLF', 'XLE', 'XLI', 'XLU']
10001110    26 shots    ['XLF', 'XLE', 'XLV', 'XLU']
11100100    25 shots    ['XLE', 'XLP', 'XLY', 'XLU']
11001001    25 shots    ['XLK', 'XLV', 'XLY', 'XLU']
11000011    25 shots    ['XLK', 'XLF', 'XLY', 'XLU']
00111010    24 shots    ['XLF', 'XLV', 'XLI', 'XLP']


In [7]:
if os.path.exists("results/RESULT_QAOA.json"):
    q = R("RESULT_QAOA.json"); print("H3 QAOA solver:", q["H3"]); print("H4 portfolio:", q["H4"])
    display(pd.DataFrame(q["performance"]).T.round(3))
else:
    print("run_qaoa_portfolio.py has not finished yet (152 sequential 18-qubit QAOA runs)")
from IPython.display import Image
display(Image("figures/fig_portfolios.png"))

H3 QAOA solver: {'success_rate': 0.20394736842105263, 'successes': 31, 'months': 152, 'random_baseline': 0.28444385582504983, 'binom_p': 0.9907705467626474, 'mean_p_opt': 0.00026581056400361064, 'uniform_p_opt': 3.814697265625e-06, 'mean_p_feasible': 0.6822576775864363, 'uniform_p_feasible': 0.0116729736328125, 'mean_approx_ratio': 0.968749720690333, 'flagged_months': 0, 'verdict': 'FAIL'}
H4 portfolio: {'sharpe_qaoa': 0.7006526593711886, 'sharpe_ew': 0.8813083178977777, 'diff': -0.18065565852658916, 'ci90': [-0.4591448706818971, 0.08680351731551052], 'verdict': 'CLOSED-NEG'}


,sharpe,ann_return,ann_vol,max_drawdown
qaoa,0.701,0.083,0.126,-0.231
bruteforce_opt,0.666,0.077,0.124,-0.172
greedy_mu,0.759,0.096,0.132,-0.167
prophet_top4,0.780,0.106,0.142,-0.219
momentum_top4,0.845,0.110,0.134,-0.146
equal_weight_18,0.881,0.104,0.121,-0.197


![output](figures/notebook/cell12_2.png)

## 5b. Regimes, 9 sector ETFs (9 qubits), and POMDP / GRPO agents
* **Regimes**: volatility regime (18 ETFs), bull/bear month (SPY), risk-on/off (SPY vs TLT),
  against persistence, VIX, Prophet and Markov-switching baselines.
* **9 sector SPDRs** shrink the QAOA problem to 9 qubits (84 valid portfolios).
* **POMDP** belief agent (2-regime HMM filter) and **GRPO** (group-relative policy optimisation,
  Plackett-Luce policy, 5 seeds, shuffled-reward control) on the same top-K decision.

In [8]:
from IPython.display import Image
rg, sc, rl = R("RESULT_REGIME.json"), R("RESULT_SECTOR.json"), R("RESULT_RL.json")
print("Regime quantum-vs-RBF test:", rg["RG_H1"])
print("9-qubit QAOA:", {k: sc["qaoa_9q"][k] for k in ("S1_argmax_rate", "S2_mean_p_opt", "S3_best16_rate", "S3_random_baseline", "verdict")})
print("GRPO / POMDP Sharpe (18 ETFs):", {k: round(rl["U18"]["portfolio"][k]["sharpe"], 2) for k in ("grpo", "pomdp", "equal_weight")})
display(Image("figures/fig_regime.png")); display(Image("figures/fig_sector_rl.png"))

Regime quantum-vs-RBF test: {'gain_pt': {'RG1': 0.07309941520467814, 'RG2': 0.0, 'RG3': -7.894736842105263}, 'p_holm': {'RG1': 1.0, 'RG2': 1.0, 'RG3': 0.05579442055794421}, 'verdict': 'CLOSED-NEG'}
9-qubit QAOA: {'S1_argmax_rate': 0.9802631578947368, 'S2_mean_p_opt': 0.010780178747724595, 'S3_best16_rate': 0.09210526315789473, 'S3_random_baseline': 0.17437867653453532, 'verdict': 'FAIL'}
GRPO / POMDP Sharpe (18 ETFs): {'grpo': 0.79, 'pomdp': 0.69, 'equal_weight': 0.88}


![output](figures/notebook/cell14_1.png)

![output](figures/notebook/cell14_2.png)

## 6. Recursive self-improvement of Prophet (RSI)
*Stage Q:* Prophet tunes its own configuration (6 on/off knobs = 6 qubits): fit a
quadratic surrogate to the configurations tried so far, let QAOA propose the next one,
repeat (24 evaluations of 64). *Stage K:* a quantum-kernel ridge model learns Prophet's
own residuals and adds a correction; a new round is kept only if the 2009-2013 error falls.
Then the improved forecaster is scored once on 2014-2026 against plain Prophet.

In [9]:
r = R("RESULT_RSI.json"); land = pd.Series(r["stage_Q"]["landscape"]).sort_values()
ax = land.plot.bar(figsize=(14, 3), color=["crimson" if k == r["stage_Q"]["base"] else "#7b3fe4" for k in land.index])
ax.set_ylabel("dev MAE"); ax.set_ylim(land.min() * 0.95, land.quantile(0.9)); ax.set_title("all 64 Prophet configurations (red = original)"); plt.show()
print("search loops:", json.dumps(r["stage_Q"]["loops"], indent=1))
print("stage K:", json.dumps(r["stage_K"], indent=1, default=str))
display(pd.DataFrame(r["test_metrics"]).T.round(4)); print("H5:", r["H5"])

![output](figures/notebook/cell16_0.png)

search loops: {
 "qaoa_seed0_best": "001000",
 "qaoa": {
  "found_dev_best_rate": 0.38,
  "mean_best_dev_mae": 0.05881170281738071
 },
 "exact_seed0_best": "001000",
 "exact": {
  "found_dev_best_rate": 0.38,
  "mean_best_dev_mae": 0.05881170281738071
 },
 "random": {
  "found_dev_best_rate": 0.32,
  "analytic": 0.375,
  "mean_best_dev_mae": 0.05825252279146648
 }
}
stage K: {
 "quantum": {
  "alpha": 10.0,
  "alpha_trials": {
   "0.1": 0.07250188031597421,
   "1.0": 0.0642140262689415,
   "10.0": 0.05836471597208683
  },
  "rounds": [
   {
    "round": 1,
    "dev_mae_before": 0.05856023574833895,
    "dev_mae_after": 0.05836471597208683,
    "accepted": true
   },
   {
    "round": 2,
    "dev_mae_before": 0.05836471597208683,
    "dev_mae_after": 0.061544775052036885,
    "accepted": false
   }
  ],
  "accepted_rounds": 1
 },
 "rbf": {
  "alpha": 10.0,
  "alpha_trials": {
   "0.1": 0.05464470720758864,
   "1.0": 0.05311831989067836,
   "10.0": 0.05208233553651609
  },
  "rounds": [


,mae,direction_hit,mean_ic
prophet_base,0.0568,0.5121,0.0069
prophet_carried_config,0.0514,0.5157,0.0089
rsi_quantum,0.0506,0.5270,-0.0074
rsi_rbf_twin,0.0452,0.5296,0.0090
random_walk_zero,0.0379,0.0007,NaN
historical_mean,0.0371,0.5881,0.0727


H5: {'per_arm': {'rsi_quantum': {'mae_diff_vs_base': -0.006204133889720374, 'p_signflip': 0.007399260073992601}, 'rsi_rbf_twin': {'mae_diff_vs_base': -0.011595195283995633, 'p_signflip': 9.999000099990002e-05}, 'prophet_carried_config': {'mae_diff_vs_base': -0.005434454070877023, 'p_signflip': 0.0235976402359764}, 'random_walk_zero': {'mae_diff_vs_base': -0.018917841459516676, 'p_signflip': 9.999000099990002e-05}}, 'verdict': 'POS', 'overfit_gap': {'dev_gain': 0.005637874781810277, 'test_gain': 0.006204133889720374}}


## 7. Live forecast (latest close, fresh data) — not investment advice

In [10]:
import glob
live = sorted(glob.glob("results/LIVE_*.md"))
from IPython.display import Markdown; display(Markdown(open(live[-1]).read()) if live else "run `python run_live.py` first")

# Live forecast as of 2026-10-23 (close) — NOT investment advice

| ETF | close | Prophet 30d | 80 % interval | signal | QSVM rank | RBF rank |
|---|---:|---:|---|---|---:|---:|
| XLU | 42.58 | -6.36 % | -8.3 .. -4.6 % | BEARISH | 1 | 12 |
| XLK | 183.24 | +5.09 % | +2.7 .. +7.5 % | BULLISH | 2 | 10 |
| EEM | 62.02 | +2.89 % | +0.3 .. +5.4 % | NEUTRAL | 3 | 16 |
| TLT | 81.34 | -6.38 % | -7.7 .. -5.1 % | BEARISH | 4 | 6 |
| QQQ | 704.39 | +3.62 % | +1.6 .. +5.6 % | BULLISH | 5 | 11 |
| GLD | 366.60 | -4.95 % | -8.2 .. -1.7 % | BEARISH | 6 | 15 |
| VNQ | 92.69 | -5.39 % | -7.1 .. -3.6 % | BEARISH | 7 | 7 |
| IEF | 91.05 | -2.57 % | -3.1 .. -2.0 % | NEUTRAL | 8 | 18 |
| XLI | 164.61 | -3.22 % | -5.2 .. -1.1 % | BEARISH | 9 | 2 |
| XLF | 55.90 | -0.75 % | -3.3 .. +1.7 % | NEUTRAL | 10 | 4 |
| XLV | 179.26 | +2.92 % | +1.5 .. +4.4 % | NEUTRAL | 11 | 3 |
| IWM | 275.72 | -6.31 % | -8.6 .. -4.0 % | BEARISH | 12 | 1 |
| XLY | 113.11 | -6.20 % | -8.8 .. -3.7 % | BEARISH | 13 | 14 |
| XLE | 69.33 | +1.75 % | -0.1 .. +3.8 % | NEUTRAL | 14 | 17 |
| XLP | 88.27 | -2.95 % | -4.3 .. -1.6 % | NEUTRAL | 15 | 8 |
| SPY | 759.91 | +1.84 % | +0.4 .. +3.3 % | NEUTRAL | 16 | 13 |
| EFA | 102.63 | -2.19 % | -4.0 .. -0.4 % | NEUTRAL | 17 | 9 |
| XLB | 51.30 | -7.20 % | -9.4 .. -5.0 % | BEARISH | 18 | 5 |

QAOA top-4 (equal weight): XLU, EEM, TLT, IEF (brute-force optimum NOT matched: XLK, XLU, EEM, TLT, P(optimum) in the QAOA state 0.000)

Backtest 2014-01..2026-08: Prophet IC 0.007 (p=0.81), top-half accuracy QSVM 49.7 % vs RBF 52.3 % (coin = 50 %): these ranks have no demonstrated predictive value. Not investment advice.


## 8. Verdicts (read from the result files)

In [11]:
qa = R("RESULT_QAOA.json") if os.path.exists("results/RESULT_QAOA.json") else None
v = dict(H1_prophet_informative=R("RESULT.json")["H1"]["verdict"], H2_qsvm_vs_rbf=R("RESULT.json")["H2"]["verdict"],
         H3_qaoa_finds_optimum=qa["H3"]["verdict"] if qa else "running", H4_portfolio_vs_equal_weight=qa["H4"]["verdict"] if qa else "running",
         H5_self_improvement=R("RESULT_RSI.json")["H5"]["verdict"])
display(pd.Series(v, name="verdict").to_frame())
display(Image("figures/fig_stock_classifiers.png")); display(Image("figures/fig_self_improvement.png"))

,verdict
H1_prophet_informative,CLOSED-NEG
H2_qsvm_vs_rbf,CLOSED-NEG
H3_qaoa_finds_optimum,FAIL
H4_portfolio_vs_equal_weight,CLOSED-NEG
H5_self_improvement,POS


![output](figures/notebook/cell20_1.png)

![output](figures/notebook/cell20_2.png)

**How to read this.** A CLOSED-NEG verdict is a real result, not a failure of the
notebook: it says the pre-registered effect was not there. All quantum parts here are
exact classical simulations (4-18 qubits), so even a positive result would be a
statement about a kernel or a heuristic, never a quantum advantage.